# GSM8K Rollout Lab — Gemma 3 & Qwen3

Browse all 1319 GSM8K test problems with their **32 packaged rollouts** from `google/gemma-3-1b-it`, **edit a problem** (with a new gold answer), and **re-run rollouts through the exact OLMES pipeline** that produced the originals (same task config, prompt template, chat formatting, sampling settings, and scoring).

Re-rolls aren't limited to the original model: pick from **Gemma 3 (1B/4B/12B)** or **Qwen3 (1.7B/4B/8B/14B, hybrid thinking)** presets — or any Hugging Face model id — and optionally set a **system prompt**.

**Requirements**
- A GPU runtime (`Runtime → Change runtime type → GPU`). The 1B default runs on any Colab GPU. Larger models need more memory — see the sizing note in step 4 (≤4B on a T4; up to ~12–14B on an A100/L4). On T4/V100 models run in fp32 (no bf16 there, and Gemma 3 in vLLM disallows fp16).
- **No Hugging Face token needed.** All presets are ungated unsloth mirrors (identical weights). If you prefer the official gated `google/gemma-3-1b-it`, set `HF_TOKEN` (see steps 2 and 4).

**Notes on replication**: new rollouts use the identical OLMES code path and settings as the packaged ones, but sampling at temperature 0.6 is stochastic — expect the same *distribution*, not identical strings. An optional seed makes your own runs repeatable (the original runs were unseeded).


## 1. Setup (~5–10 min on a fresh VM)

Clones the repo and installs the vendored OLMES harness with vLLM. **On the first run this cell will automatically restart the runtime once the install finishes** (the freshly installed packages only load in a fresh kernel) — that's expected. When it reconnects, just re-run from step 2.


In [ ]:
import os
from pathlib import Path

# NOTE: the public repo name is "gsm8k-rollut-lab" (the GitHub repo has a typo
# in its name). It is public, so no GitHub credentials/token are needed.
REPO_URL = "https://github.com/prudhvirajn/gsm8k-rollut-lab"
REPO_DIR = Path("/content/gsm8k-rollout-lab")

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {str(REPO_DIR)}
%cd {str(REPO_DIR)}

# Install once, then auto-restart the kernel: the editable-install .pth and the
# freshly (re)installed torch/vllm are only picked up by a fresh Python process.
# The marker file makes this a no-op when you re-run after the restart.
_marker = Path("/content/.olmes_installed")
if not _marker.exists():
    !pip install -q -e "./olmes[gpu]"
    _marker.touch()
    print("OLMES installed — restarting the runtime. After it reconnects, "
          "re-run from step 2 (skip this cell).")
    os.kill(os.getpid(), 9)  # Colab auto-restarts the kernel; disk state persists


## 2. Environment (no Hugging Face token needed)

This notebook uses the **ungated** mirror `unsloth/gemma-3-1b-it` (identical Gemma 3 1B-IT weights), so you do **not** need a Hugging Face token or to accept the Gemma license on the Hub.

If you would rather use the official gated `google/gemma-3-1b-it`, set `HF_TOKEN` as a Colab secret and change the model in step 4 — see the comment in that cell.


In [ ]:
import os

# The default model (unsloth/gemma-3-1b-it) is ungated -> no HF token required.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Keep the interface clean: silence verbose logging from vLLM / HF datasets /
# transformers (these env vars must be set before those libs import, i.e. step 4).
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["DATASETS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Optional: only needed if you switch to the gated google/gemma-3-1b-it in step 4.
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


## 3. Load the packaged rollouts and browse

Filter by passrate (e.g. set the range to 0–0.3 to look at hard problems) or search the question text, then step through the 32 rollouts of the selected problem.


In [ ]:
import sys
sys.path.insert(0, "/content/gsm8k-rollout-lab")

from gsm8k_rollout_lab import load_bundle, ProblemBrowser

problems = load_bundle()
print(f"{len(problems)} problems, {sum(p['n_rollouts'] for p in problems)} rollouts")

browser = ProblemBrowser(problems)
browser.display()


## 4. Load the model (once, ~2 min)

Pick a model with `MODEL` below — Gemma 3 (1B/4B/12B) or Qwen3 (1.7B/4B/8B/14B, hybrid *thinking* models), or paste any Hugging Face id. `build_model_overrides` auto-selects dtype and engine settings for your GPU (bf16 on A100/L4, fp32 on T4/V100).

**Sizing:** a single Colab **A100/L4 is ~40 GB**, so in bf16 you can run up to ~12–14B (Gemma 3 12B, Qwen3 14B). A **T4 (16 GB)** realistically handles the 1B–4B models in fp32. Larger models need quantization or multiple GPUs.

The vLLM model stays resident, so each re-run below only pays generation time. **Switching `MODEL` means reloading** — re-run this cell (and on tight GPUs, restart the runtime first so the old model's memory is freed).


In [ ]:
from gsm8k_rollout_lab import GSM8KRolloutRunner, build_model_overrides, MODEL_PRESETS

# Pick a preset name below, or pass ANY Hugging Face model id to build_model_overrides.
# All presets are ungated unsloth mirrors -> no HF token needed.
for name, p in MODEL_PRESETS.items():
    print(f"  {name:14s} {p['note']}")

MODEL = "gemma-3-1b-it"   # <- a preset key above, or e.g. "unsloth/Qwen3-4B", or any HF id

# build_model_overrides picks dtype + engine settings from the detected GPU:
#   A100/L4/H100 (capability >= 8): bf16 + FlashAttention (vLLM default).
#   T4/V100 (capability < 8): float32 + enforce_eager + the Triton attention
#     backend (avoids the FlexAttention crash Gemma 3 hits on those GPUs).
# Override anything explicitly if you like, e.g. build_model_overrides(MODEL, max_num_seqs=4).
overrides = build_model_overrides(MODEL)
print(f"\nLoading {overrides['model']}  (dtype={overrides.get('dtype')}, "
      f"gpu_mem={overrides.get('gpu_memory_utilization')})")

runner = GSM8KRolloutRunner(model_overrides=overrides)


## 5. Edit a problem and run rollouts

`Load selected problem` pulls the problem chosen in the browser above into the form. Edit the question, set the **new gold answer** (just the final number), pick generation settings, and hit `Run rollouts`. The result view shows the new passrate next to the original problem's, and you can step through the new completions.

- **system** — an optional system prompt (the original runs used none).
- **thinking** — for hybrid reasoning models (Qwen3): leave on *model default*, or force `/think` / `/no_think`. Thinking models reason inside `<think>…</think>` first, so give them a larger **max toks** (and expect longer runs). The final boxed answer is still scored the same way.

Leave `seed` blank to match the original (unseeded) setup; set it to make a run repeatable (seeded runs generate one rollout per seed, which is slower).


In [ ]:
from gsm8k_rollout_lab import EditRunPanel

panel = EditRunPanel(runner, browser)
panel.display()


### Programmatic API (optional)

The same runner can be called directly — useful for sweeping several edits:


In [ ]:
# result = runner.run(
#     question="Janet's ducks lay 20 eggs per day. ...",
#     answer="26",                 # bare final answer, or full CoT ending in '#### 26'
#     n_rollouts=32,
#     temperature=0.6,
#     top_p=0.95,
#     seed=None,
#     system_prompt=None,          # optional chat system message
#     thinking=None,               # Qwen3: None=default, True=/think, False=/no_think
# )
# print(result["n_correct"], "/", result["n_rollouts"])


## 6. Save your runs

Every run is written to `rollout_runs/<timestamp>_<name>/` in native OLMES format (`task-000-gsm8k-predictions.jsonl`, `metrics.json`, the task spec, and a scored `result.json`), plus a compact `rollout_runs/history.jsonl`. Colab VMs are ephemeral — copy them to Drive before closing:


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r rollout_runs /content/drive/MyDrive/gsm8k_rollout_lab_runs
